# Vera CoT Verifier — LLM-AggreFact Benchmark Evaluation

This notebook evaluates Vera's **fine-tuned LFM2.5-1.2B CoT verifier** against the [LLM-AggreFact benchmark](https://huggingface.co/datasets/lytang/LLM-AggreFact) (Tang et al., EMNLP 2024 / MiniCheck).

The CoT verifier generates step-by-step reasoning before producing a verdict, unlike the zero-shot ModernBERT verifier.

- **Labels**: 1 = supported, 0 = unsupported
- **Primary Metric**: Balanced Accuracy (BAcc)
- **Model**: Fine-tuned `LFM2.5-1.2B-Instruct` (1.2B params, LoRA)

## 1. Setup

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.57.3
!pip install --no-deps trl==0.22.2
!pip install scikit-learn pandas tabulate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = 'drive/MyDrive/vera/verification'

RESULTS_DIR = 'drive/MyDrive/vera/evaluations/llmaggrefact-cot-verifier'

Mounted at /content/drive


In [ ]:
HF_TOKEN = None  # <-- paste your token here


from huggingface_hub import login as hf_login

if HF_TOKEN:
    hf_login(token=HF_TOKEN)
    print("✅ Authenticated with HuggingFace.")
else:
    print("⚠️  No HF token set — will fail on gated datasets. Paste your token above.")

✅ Authenticated with HuggingFace.


## 2. Load Benchmark

In [ ]:
from datasets import load_dataset
import pandas as pd

SPLIT = "test"  # Change to "dev" for the development set

print(f"Loading LLM-AggreFact benchmark ({SPLIT} split)...")
benchmark = load_dataset("lytang/LLM-AggreFact", split=SPLIT, token=HF_TOKEN)
print(f"Total samples: {len(benchmark)}")

df_bench = benchmark.to_pandas()
print("\nSamples per dataset:")
print(df_bench['dataset'].value_counts().to_string())

Loading LLM-AggreFact benchmark (test split)...


README.md:   0%|          | 0.00/4.93k [00:00<?, ?B/s]

data/dev-00000-of-00001.parquet:   0%|          | 0.00/35.5M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

Generating dev split:   0%|          | 0/30420 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/29320 [00:00<?, ? examples/s]

Total samples: 29320

Samples per dataset:
dataset
RAGTruth           16371
ExpertQA            3702
Lfqa                1911
Reveal              1710
FactCheck-GPT       1566
ClaimVerify         1088
TofuEval-MeetB       772
TofuEval-MediaS      726
AggreFact-CNN        558
AggreFact-XSum       558
Wice                 358


## 3. Load Fine-Tuned CoT Verifier

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "werstal/vera-verifier-cot-lfm2.5-1.2b"


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=False,
    load_in_8bit=False,
    load_in_16bit=True,
    full_finetuning=False,
)

FastLanguageModel.for_inference(model)
print("✅ CoT Verifier loaded!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Lfm2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.34G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ CoT Verifier loaded!


## 4. Inference Helpers

In [ ]:
import json
from sklearn.metrics import (
    balanced_accuracy_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

# System prompt — identical to training notebook and deployment/api/main.py
STUDENT_PROMPT = """You are a fact-checker. Verify claims against evidence.

TASK: Given a CLAIM and EVIDENCE snippets, determine if the claim is supported or refuted.

OUTPUT FORMAT (JSON):
{
  "reasoning": "<analyze each evidence piece, then conclude>",
  "verdict": "SUPPORTED" | "REFUTED"
}

RULES:
1. SUPPORTED = claim confirmed by evidence
2. REFUTED = evidence contradicts the claim
3. Reference evidence by number: "Evidence 1 shows..."
4. If any key part is contradicted, verdict is REFUTED"""


def format_input(claim, evidence_list):
    """Format claim and evidence for the model.
    Matches the format_input function from the training notebook.
    Evidence is presented as a numbered list.
    """
    lines = [f"CLAIM: {claim}", "", "EVIDENCE:"]
    for i, e in enumerate(evidence_list, 1):
        lines.append(f"[{i}] {e}")
    return "\n".join(lines)


def parse_json_safe(text):
    """Parse JSON from model output. Returns (parsed_dict, is_valid).
    Matches the parse_json_safe function from the training notebook.
    """
    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            return parsed, True
        return {}, False
    except (json.JSONDecodeError, ValueError, TypeError):
        try:
            start = text.find('{')
            end = text.rfind('}') + 1
            if start >= 0 and end > start:
                parsed = json.loads(text[start:end])
                return parsed, True
        except:
            pass
    return {}, False


def extract_verdict(raw_output):
    """Extract verdict from model output, with fallback parsing."""
    parsed, is_valid = parse_json_safe(raw_output)

    if is_valid and 'verdict' in parsed:
        verdict = parsed['verdict'].upper().strip()
        reasoning = parsed.get('reasoning', '')
        if verdict in ['SUPPORTED', 'REFUTED']:
            return verdict, reasoning, True

    upper = raw_output.upper()
    if 'SUPPORTED' in upper:
        return 'SUPPORTED', '', False
    elif 'REFUTED' in upper:
        return 'REFUTED', '', False

    return 'REFUTED', '', False


def compute_metrics(y_true, y_pred):
    """Compute evaluation metrics."""
    bacc = balanced_accuracy_score(y_true, y_pred)
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "BAcc": round(bacc * 100, 1),
        "Accuracy": round(acc * 100, 1),
        "Precision": round(prec * 100, 1),
        "Recall": round(rec * 100, 1),
        "F1": round(f1 * 100, 1),
        "TP": int(tp), "TN": int(tn),
        "FP": int(fp), "FN": int(fn),
        "N": len(y_true),
        "Pos%": round(sum(y_true) / len(y_true) * 100, 1),
    }

## 5. Batched Generation

In [ ]:
EVAL_BATCH_SIZE = 256

In [ ]:
def batched_generate(claims_and_docs, system_prompt, curr_model, curr_tokenizer,
                     batch_size=8, max_new_tokens=512):
    """Batched inference matching the training notebook's approach.

    Args:
        claims_and_docs: list of (claim, doc) tuples
        system_prompt: system prompt string
        curr_model: the model to use
        curr_tokenizer: the tokenizer to use
        batch_size: batch size for generation
        max_new_tokens: max tokens to generate

    Returns:
        list of raw model output strings
    """
    all_outputs = []

    curr_tokenizer.padding_side = "left"
    if not curr_tokenizer.pad_token:
        curr_tokenizer.pad_token = curr_tokenizer.eos_token

    for i in range(0, len(claims_and_docs), batch_size):
        batch = claims_and_docs[i:i+batch_size]

        batch_messages = [
            [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": format_input(claim, [doc])}
            ]
            for claim, doc in batch
        ]

        prompts = [
            curr_tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
            for msg in batch_messages
        ]

        inputs = curr_tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2048
        ).to("cuda")

        outputs = curr_model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            temperature=0.1,
            do_sample=True,
            pad_token_id=curr_tokenizer.pad_token_id,
            stop_strings=["}", "}\n"],
            tokenizer=curr_tokenizer
        )

        prompt_length = inputs.input_ids.shape[1]
        generated_tokens = outputs[:, prompt_length:]
        decoded_batch = curr_tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        all_outputs.extend(decoded_batch)

    return all_outputs

In [ ]:
print("Running sanity test...")
test_outputs = batched_generate(
    [("The Eiffel Tower is in Paris.",
      "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France.")],
    STUDENT_PROMPT, model, tokenizer, batch_size=1
)
print(f"Raw output: {test_outputs[0][:300]}")
verdict, reasoning, is_valid = extract_verdict(test_outputs[0])
print(f"Verdict: {verdict}, Valid JSON: {is_valid}")
print("✅ Sanity test passed!" if verdict in ['SUPPORTED', 'REFUTED'] else '⚠️ Check model output')

Running sanity test...
Raw output: {"reasoning": "The claim states that the Eiffel Tower is located in Paris. Evidence 1 confirms that the Eiffel Tower is a wrought-iron lattice tower situated on the Champ de Mars in Paris, France. The evidence directly supports the location and existence of the claim.", "verdict": "SUPPORTED"}
Verdict: SUPPORTED, Valid JSON: True
✅ Sanity test passed!


## 6. Run Evaluation

In [ ]:
print(df_bench.groupby('dataset')['doc'].apply(lambda x: x.str.len().max()).sort_values(ascending=False).to_string())

dataset
ClaimVerify        120274
ExpertQA           117519
Wice                93864
AggreFact-CNN       10572
AggreFact-XSum       9953
RAGTruth             9918
TofuEval-MediaS      5906
TofuEval-MeetB       5772
Lfqa                 2988
Reveal               2822
FactCheck-GPT        1028


In [ ]:
import time
import os
import gc
from tqdm.notebook import tqdm
from datetime import datetime

ALL_DATASETS = [
    "AggreFact-CNN", "AggreFact-XSum",
    "TofuEval-MediaS", "TofuEval-MeetB",
    "Wice", "Reveal", "ClaimVerify",
    "FactCheck-GPT", "ExpertQA", "Lfqa", "RAGTruth",
]

TARGET_DATASETS = ALL_DATASETS  # Change to e.g. ["Wice", "FactCheck-GPT"] for a quick test
MAX_SAMPLES = None              # Set to e.g. 50 for a quick smoke test

all_results = {}
all_predictions = []
overall_y_true = []
overall_y_pred = []
total_time = 0
total_json_valid = 0
total_json_invalid = 0

for ds_name in TARGET_DATASETS:
    ds_samples = df_bench[df_bench['dataset'] == ds_name]

    if len(ds_samples) == 0:
        print(f"⚠ No samples found for '{ds_name}', skipping.")
        continue

    if MAX_SAMPLES and len(ds_samples) > MAX_SAMPLES:
        ds_samples = ds_samples.head(MAX_SAMPLES)

    print(f"\n{'='*60}")
    print(f"Evaluating: {ds_name} ({len(ds_samples)} samples)")
    print(f"{'='*60}")

    torch.cuda.empty_cache()
    gc.collect()

    claims_and_docs = [(row['claim'], row['doc']) for _, row in ds_samples.iterrows()]

    avg_doc_len = ds_samples['doc'].str.len().mean()
    effective_batch_size = EVAL_BATCH_SIZE

    start_time = time.time()

    raw_outputs = batched_generate(
        claims_and_docs, STUDENT_PROMPT, model, tokenizer,
        batch_size=effective_batch_size
    )

    elapsed = time.time() - start_time
    total_time += elapsed

    y_true, y_pred = [], []
    ds_json_valid = 0
    ds_json_invalid = 0

    for (_, row), raw_output in zip(ds_samples.iterrows(), raw_outputs):
        verdict, reasoning, is_valid_json = extract_verdict(raw_output)

        if is_valid_json:
            ds_json_valid += 1
        else:
            ds_json_invalid += 1

        pred_label = 1 if verdict == "SUPPORTED" else 0

        y_true.append(row['label'])
        y_pred.append(pred_label)

        all_predictions.append({
            "dataset": ds_name,
            "claim": row['claim'][:200],
            "doc_len": len(row['doc']),
            "gold_label": int(row['label']),
            "pred_label": pred_label,
            "verdict": verdict,
            "reasoning": reasoning[:300],
            "raw_output": raw_output[:500],
            "valid_json": is_valid_json,
            "correct": int(row['label']) == pred_label,
        })

    total_json_valid += ds_json_valid
    total_json_invalid += ds_json_invalid

    metrics = compute_metrics(y_true, y_pred)
    metrics["Time (s)"] = round(elapsed, 1)
    metrics["Throughput (ex/s)"] = round(len(ds_samples) / elapsed, 1)
    json_valid_rate = round(ds_json_valid / len(ds_samples) * 100, 1)
    metrics["JSON Valid %"] = json_valid_rate
    all_results[ds_name] = metrics

    overall_y_true.extend(y_true)
    overall_y_pred.extend(y_pred)

    print(f"  BAcc: {metrics['BAcc']}%  |  Acc: {metrics['Accuracy']}%  |  "
          f"F1: {metrics['F1']}%  |  N: {metrics['N']}  |  "
          f"JSON: {json_valid_rate}%  |  Time: {metrics['Time (s)']}s")

if overall_y_true:
    overall_metrics = compute_metrics(overall_y_true, overall_y_pred)
    overall_metrics["Time (s)"] = round(total_time, 1)
    overall_metrics["Throughput (ex/s)"] = round(len(overall_y_true) / total_time, 1)
    overall_metrics["JSON Valid %"] = round(total_json_valid / len(overall_y_true) * 100, 1)
    all_results["OVERALL"] = overall_metrics

print(f"\n✅ Evaluation complete!")
print(f"JSON valid: {total_json_valid}/{len(overall_y_true)} "
      f"({total_json_valid/len(overall_y_true)*100:.1f}%)")


Evaluating: AggreFact-CNN (558 samples)
  BAcc: 52.1%  |  Acc: 72.6%  |  F1: 83.6%  |  N: 558  |  JSON: 98.9%  |  Time: 148.5s

Evaluating: AggreFact-XSum (558 samples)
  BAcc: 57.6%  |  Acc: 57.9%  |  F1: 63.6%  |  N: 558  |  JSON: 99.8%  |  Time: 67.0s

Evaluating: TofuEval-MediaS (726 samples)
  BAcc: 56.2%  |  Acc: 63.5%  |  F1: 74.5%  |  N: 726  |  JSON: 100.0%  |  Time: 72.6s

Evaluating: TofuEval-MeetB (772 samples)
  BAcc: 64.3%  |  Acc: 79.9%  |  F1: 87.8%  |  N: 772  |  JSON: 100.0%  |  Time: 93.4s

Evaluating: Wice (358 samples)
  BAcc: 57.6%  |  Acc: 57.5%  |  F1: 45.7%  |  N: 358  |  JSON: 73.2%  |  Time: 84.0s

Evaluating: Reveal (1710 samples)
  BAcc: 80.6%  |  Acc: 76.2%  |  F1: 63.6%  |  N: 1710  |  JSON: 100.0%  |  Time: 92.3s

Evaluating: ClaimVerify (1088 samples)
  BAcc: 57.2%  |  Acc: 61.5%  |  F1: 71.5%  |  N: 1088  |  JSON: 72.7%  |  Time: 234.4s

Evaluating: FactCheck-GPT (1566 samples)
  BAcc: 67.7%  |  Acc: 66.6%  |  F1: 50.1%  |  N: 1566  |  JSON: 99.7%  | 

## 7. Results Summary

In [ ]:
results_df = pd.DataFrame(all_results).T
display_cols = ["BAcc", "Accuracy", "Precision", "Recall", "F1", "N", "Pos%",
                "JSON Valid %", "Time (s)"]

print("=" * 90)
print("RESULTS — Vera CoT Verifier (LFM2.5-1.2B) on LLM-AggreFact")
print(f"Split: {SPLIT}  |  Model: {MODEL_PATH}")
print("=" * 90)
results_df[display_cols]

RESULTS — Vera CoT Verifier (LFM2.5-1.2B) on LLM-AggreFact
Split: test  |  Model: drive/MyDrive/vera/verification/model_output/lora_adapters


,BAcc,Accuracy,Precision,Recall,F1,N,Pos%,JSON Valid %,Time (s)
AggreFact-CNN,52.1,72.6,90.3,77.8,83.6,558.0,89.8,98.9,148.5
AggreFact-XSum,57.6,57.9,56.9,71.9,63.6,558.0,51.1,99.8,67.0
TofuEval-MediaS,56.2,63.5,79.7,70.0,74.5,726.0,76.3,100.0,72.6
TofuEval-MeetB,64.3,79.9,85.9,89.9,87.8,772.0,80.6,100.0,93.4
Wice,57.6,57.5,37.9,57.7,45.7,358.0,31.0,73.2,84.0
Reveal,80.6,76.2,49.5,89.0,63.6,1710.0,23.4,100.0,92.3
ClaimVerify,57.2,61.5,77.1,66.7,71.5,1088.0,72.5,72.7,234.4
FactCheck-GPT,67.7,66.6,39.1,69.9,50.1,1566.0,24.0,99.7,88.7
ExpertQA,54.4,69.9,82.0,80.1,81.0,3702.0,80.3,95.4,593.5
Lfqa,67.3,68.8,72.3,75.8,74.0,1911.0,58.7,99.8,140.2


## 8. Comparison with Published Baselines

In [ ]:
baselines = {
    "AlignScore (355M)": {
        "AggreFact-CNN": 52.4, "AggreFact-XSum": 71.4,
        "TofuEval-MediaS": 69.2, "TofuEval-MeetB": 72.6,
        "Wice": 66.0, "Reveal": 85.3, "ClaimVerify": 69.6,
        "FactCheck-GPT": 74.3, "ExpertQA": 58.3, "Lfqa": 84.5,
    },
    "MiniCheck-FT5 (770M)": {
        "AggreFact-CNN": 69.9, "AggreFact-XSum": 74.3,
        "TofuEval-MediaS": 73.6, "TofuEval-MeetB": 77.3,
        "Wice": 72.2, "Reveal": 86.2, "ClaimVerify": 74.6,
        "FactCheck-GPT": 74.7, "ExpertQA": 59.0, "Lfqa": 85.2,
    },
    "GPT-4 (~1.8T)": {
        "AggreFact-CNN": 66.7, "AggreFact-XSum": 76.5,
        "TofuEval-MediaS": 71.4, "TofuEval-MeetB": 79.9,
        "Wice": 80.4, "Reveal": 87.8, "ClaimVerify": 67.6,
        "FactCheck-GPT": 79.9, "ExpertQA": 59.2, "Lfqa": 83.1,
    },
}

comparison_data = {}
for name, scores in baselines.items():
    row = {ds: scores.get(ds, None) for ds in ALL_DATASETS}
    row["Avg"] = round(sum(v for v in scores.values()) / len(scores), 1)
    comparison_data[name] = row

vera_row = {}
for ds in ALL_DATASETS:
    if ds in all_results:
        vera_row[ds] = all_results[ds]["BAcc"]
vera_row["Avg"] = all_results.get("OVERALL", {}).get("BAcc", None)
comparison_data["Vera CoT (1.2B)"] = vera_row

comp_df = pd.DataFrame(comparison_data).T
print("Balanced Accuracy (%) per dataset — comparison with published baselines")
print("-" * 80)
comp_df

Balanced Accuracy (%) per dataset — comparison with published baselines
--------------------------------------------------------------------------------


,AggreFact-CNN,AggreFact-XSum,TofuEval-MediaS,TofuEval-MeetB,Wice,Reveal,ClaimVerify,FactCheck-GPT,ExpertQA,Lfqa,RAGTruth,Avg
AlignScore (355M),52.4,71.4,69.2,72.6,66.0,85.3,69.6,74.3,58.3,84.5,NaN,70.4
MiniCheck-FT5 (770M),69.9,74.3,73.6,77.3,72.2,86.2,74.6,74.7,59.0,85.2,NaN,74.7
GPT-4 (~1.8T),66.7,76.5,71.4,79.9,80.4,87.8,67.6,79.9,59.2,83.1,NaN,75.2
Vera CoT (1.2B),52.1,57.6,56.2,64.3,57.6,80.6,57.2,67.7,54.4,67.3,63.0,66.7


## 9. Error Analysis

In [ ]:
preds_df = pd.DataFrame(all_predictions)

print("JSON Validity Rate by Dataset")
print("=" * 50)
json_df = preds_df.groupby('dataset').apply(
    lambda x: pd.Series({
        'total': len(x),
        'valid_json': x['valid_json'].sum(),
        'json_rate': f"{x['valid_json'].mean()*100:.1f}%"
    })
)
print(json_df.to_string())

invalid = preds_df[~preds_df['valid_json']].head(5)
if len(invalid) > 0:
    print(f"\n\nSample Invalid JSON Outputs ({len(invalid)} shown):")
    print("=" * 50)
    for _, row in invalid.iterrows():
        print(f"\nDataset: {row['dataset']}")
        print(f"Claim: {row['claim'][:100]}...")
        print(f"Verdict (fallback): {row['verdict']}")
        print(f"Raw output: {row['raw_output'][:200]}...")
        print("-" * 50)
else:
    print("\n✅ All outputs were valid JSON!")

print("\n\nVerdict Distribution")
print("=" * 50)
print(preds_df['verdict'].value_counts().to_string())

JSON Validity Rate by Dataset
                 total  valid_json json_rate
dataset                                     
AggreFact-CNN      558         552     98.9%
AggreFact-XSum     558         557     99.8%
ClaimVerify       1088         791     72.7%
ExpertQA          3702        3533     95.4%
FactCheck-GPT     1566        1562     99.7%
Lfqa              1911        1907     99.8%
RAGTruth         16371       16122     98.5%
Reveal            1710        1710    100.0%
TofuEval-MediaS    726         726    100.0%
TofuEval-MeetB     772         772    100.0%
Wice               358         262     73.2%


Sample Invalid JSON Outputs (5 shown):

Dataset: AggreFact-CNN
Claim: Matt Phillips headed Queens Park Rangers into a seventh - minute lead. Christian Benteke equalised f...
Verdict (fallback): SUPPORTED
Raw output: ers were fighting for the last point in a thrilling contest.

Source: C. Pawson.

"Evidence 1: Evidence 1 shows that the match was an emotional rollercoaster... It sho

/tmp/ipython-input-1985775463.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  json_df = preds_df.groupby('dataset').apply(


## 10. Save Results to Drive

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

metrics_file = os.path.join(RESULTS_DIR, f"metrics_{SPLIT}_{timestamp}.json")
with open(metrics_file, "w") as f:
    json.dump({
        "model": MODEL_PATH,
        "model_type": "CoT Verifier (LFM2.5-1.2B fine-tuned)",
        "split": SPLIT,
        "timestamp": timestamp,
        "datasets": TARGET_DATASETS,
        "json_valid_total": total_json_valid,
        "json_invalid_total": total_json_invalid,
        "per_dataset": all_results,
    }, f, indent=2)
print(f"✅ Metrics saved to: {metrics_file}")

preds_file = os.path.join(RESULTS_DIR, f"predictions_{SPLIT}_{timestamp}.jsonl")
with open(preds_file, "w") as f:
    for pred in all_predictions:
        f.write(json.dumps(pred) + "\n")
print(f"✅ Predictions saved to: {preds_file}")

csv_file = os.path.join(RESULTS_DIR, f"results_{SPLIT}_{timestamp}.csv")
results_df.to_csv(csv_file)
print(f"✅ Results CSV saved to: {csv_file}")

comp_csv = os.path.join(RESULTS_DIR, f"comparison_{SPLIT}_{timestamp}.csv")
comp_df.to_csv(comp_csv)
print(f"✅ Comparison CSV saved to: {comp_csv}")

✅ Metrics saved to: drive/MyDrive/vera/evaluations/llmaggrefact-cot-verifier/metrics_test_20260224_015013.json
✅ Predictions saved to: drive/MyDrive/vera/evaluations/llmaggrefact-cot-verifier/predictions_test_20260224_015013.jsonl
✅ Results CSV saved to: drive/MyDrive/vera/evaluations/llmaggrefact-cot-verifier/results_test_20260224_015013.csv
✅ Comparison CSV saved to: drive/MyDrive/vera/evaluations/llmaggrefact-cot-verifier/comparison_test_20260224_015013.csv


In [ ]:
from google.colab import runtime
runtime.unassign()